# Task 3: Logistic Regression with GloVe Embeddings

## Objective

We convert IMDB reviews into vectors using pretrained GloVe embeddings.

Then we use Logistic Regression from sklearn for sentiment classification:

* 1 = Positive
* 0 = Negative

---

# Step 1: Import Libraries

In [39]:
import numpy as np
import pandas as pd
import json
import re

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

---

# Step 2: Load Dataset

In [40]:
df = pd.read_csv("./datasets/imdb_top_500.csv")

with open("./datasets/tiny_glove.json", "r") as f:
    glove = json.load(f)

In [41]:
print("Dataset size:", len(df))
print("Columns:", df.columns.tolist())

print("\nFirst review preview:")
print(df["text"].iloc[0][:300])

print("\nFirst label:", df["label"].iloc[0])
print("First rating:", df["rating"].iloc[0])

print("\nVocabulary size:", len(glove))

Dataset size: 500
Columns: ['text', 'label', 'rating']

First review preview:
There are lots of extremely good-looking people in this movie. That's probably the best thing about it. Perhaps that even makes it worth watching.<br /><br />"Loaded" tells the story of Tristan Price (Jesse Metcalfe), a young man who's about to make his mark on the world. He's the son of a well-to-d

First label: 0
First rating: 4

Vocabulary size: 4993


---

# Step 3: Clean and Tokenize Text

In [42]:
def tokenize(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z\s]", "", text)
    return text.split()

In [43]:
sample_tokens = tokenize(df["text"].iloc[0])

print("First 20 tokens:")
print(sample_tokens[:20])

print("Total token count:", len(sample_tokens))

First 20 tokens:
['there', 'are', 'lots', 'of', 'extremely', 'goodlooking', 'people', 'in', 'this', 'movie', 'thats', 'probably', 'the', 'best', 'thing', 'about', 'it', 'perhaps', 'that', 'even']
Total token count: 397


---

# Step 4: Convert One Review into One Vector

In [44]:
def get_embedding(text, glove, dim=50):
    tokens = tokenize(text)

    vectors = [
        np.array(glove[word])
        for word in tokens
        if word in glove
    ]

    if len(vectors) == 0:
        return np.zeros(dim)

    return np.mean(vectors, axis=0)

In [45]:
sample_vector = get_embedding(df["text"].iloc[0], glove)

print("Embedding shape:", sample_vector.shape)

print("First 10 embedding values:")
print(sample_vector[:10])

Embedding shape: (50,)
First 10 embedding values:
[ 0.37263812  0.185661   -0.12914167 -0.13954573  0.42756589  0.18937722
 -0.45071481 -0.11065897 -0.19588969  0.03904323]


---

# Step 5: Build Features, Labels, and Original Text Together

This is important.

We keep the original review text so later predictions match the correct review.

In [46]:
X = np.array([get_embedding(text, glove) for text in df["text"]])

y = df["label"].values

texts = df["text"].values

In [47]:
print("Feature matrix shape:", X.shape)
print("Labels shape:", y.shape)
print("Texts shape:", texts.shape)

print("Positive ratio:", np.mean(y))

Feature matrix shape: (500, 50)
Labels shape: (500,)
Texts shape: (500,)
Positive ratio: 0.5


---

# Step 6: Split Train and Test Data Correctly

In [48]:
X_train, X_test, y_train, y_test, text_train, text_test = train_test_split(
    X,
    y,
    texts,
    test_size=0.2,
    random_state=42
)

In [49]:
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nFirst test review preview:")
print(text_test[0][:300])

Training samples: 400
Testing samples: 100

First test review preview:


---

# Step 7: Standardize Features

In [50]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)

X_test = scaler.transform(X_test)

In [51]:
print("Feature scaling complete.")

print("First standardized training vector:")
print(X_train[0][:10])

Feature scaling complete.
First standardized training vector:
[-0.26453417  1.07358292 -0.44928147 -0.01352123  0.54247588  0.29498412
  0.3586582   0.53818541  0.29754456 -1.63636714]


---

# Step 8: Train Logistic Regression

In [52]:
model = LogisticRegression()

model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [53]:
print("Model training complete.")

Model training complete.


---

# Step 9: Make Predictions

In [54]:
train_pred = model.predict(X_train)

test_pred = model.predict(X_test)

In [62]:
print("First 30 test predictions:")
print(test_pred[:30])

print("\nFirst 30 true labels:")
print(y_test[:30])

First 30 test predictions:
[0 1 1 1 0 0 0 1 1 1 1 1 1 0 1 1 1 1 0 0 1 0 0 1 1 0 1 1 0 1]

First 30 true labels:
[0 0 0 0 0 0 1 1 1 1 0 1 0 0 0 1 1 0 0 0 1 0 0 1 0 1 0 1 0 1]


---

# Step 10: Evaluate Accuracy

In [56]:
train_acc = accuracy_score(y_train, train_pred)

test_acc = accuracy_score(y_test, test_pred)

In [57]:
print("Train Accuracy:", train_acc)

print("Test Accuracy:", test_acc)

Train Accuracy: 0.75
Test Accuracy: 0.7


---

# Step 11: Compare Real Test Reviews with Predictions

Now each prediction matches the correct original review.

In [65]:
for i in range(85, 90):
    print(f"\nReview {i+1}")
    print("-" * 60)

    print(text_test[i][:400])

    print("\nTrue Label:", y_test[i])

    print("Predicted Label:", test_pred[i])

    if y_test[i] == test_pred[i]:
        print("Result: Correct")
    else:
        print("Result: Incorrect")


Review 86
------------------------------------------------------------
Uninspired direction leaves a decent cast stranded in a handsome but bland adaptation, in which dialogue seems recited rather than heartfelt, and cash strapped appearances by the ghosts fail to round up any sense of awe or magic; Edward Woodward, as the Ghost of Christmas Present, wobbles around on stilts and seems to be doing an impression of Bernard Cribbins. As Scrooge, George C. Scott is too w

True Label: 0
Predicted Label: 0
Result: Correct

Review 87
------------------------------------------------------------
William Russ is the main character throughout this made for TV movie. He left his family behind to only reappear and begin paying off his debts. But he tries to keep away from his family. Thats where Peter Falk (Colombo) comes in, playing several different roles, to convince him to come home.<br /><br />The story is average and they actually managed to get a former star (Peter Falk) and use him to

Tru

---

# Step 12: Predict Sentiment for New Reviews

In [59]:
sample_reviews = [
    "This movie was fantastic with brilliant acting",
    "I hated this movie it was boring and terrible",
    "The film was okay not great but not bad"
]

In [60]:
sample_X = np.array([
    get_embedding(text, glove)
    for text in sample_reviews
])

sample_X = scaler.transform(sample_X)

sample_preds = model.predict(sample_X)

In [61]:
for review, pred in zip(sample_reviews, sample_preds):
    print("\nReview:")
    print(review)

    print(
        "Predicted Sentiment:",
        "Positive" if pred == 1 else "Negative"
    )


Review:
This movie was fantastic with brilliant acting
Predicted Sentiment: Positive

Review:
I hated this movie it was boring and terrible
Predicted Sentiment: Negative

Review:
The film was okay not great but not bad
Predicted Sentiment: Negative
